# R08-H53 - the embedding bill is optional: BM25 vs Titan at graph seeding

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-06 <br>
**Pipeline stage**: R08 contrarian slate 3, first runnable entry <br>
**Graph**: rebuilt CPAP corpus (neo4j2); BM25 side needs ZERO API calls of any kind <br>

The whole R01-R07 stack assumes a dense vector index at its base. On a specification-heavy corpus (model codes, part numbers, exact feature names) lexical match is strong by construction - so BM25 over the EXACT text the entity embeddings encode (`{type}: {name} - {description[:200]}`, `embeddings.py:48`) gets a head-to-head against the Titan index on H34's metric: pure-seed evidence recall (gold strings present in the top-k seed nodes' own renders, alias merge off).

BM25 (Okapi) scores a query q against entity document d:

$$\text{score}(q,d) = \sum_{t \in q} \text{IDF}(t)\ \frac{f_{t,d}\,(k_1+1)}{f_{t,d} + k_1\,(1-b+b\,|d|/\bar{|d|})}$$

## Approach
1. **Index** - all entities from the graph, one BM25 document each, built from the identical embedding text (fairness by construction)
2. **Seed** - per probe: BM25 top-k vs Titan vector top-k (production k=8), plus the union channel (dense+sparse complementarity, the constructive consequence either way)
3. **Score** - H34's matcher (substring, unit-stripped numeric skeleton, token majority) on the pure-seed renders
4. **Verdict** - pre-registered: confirmed if BM25 lands within 10% relative of the vector channel (or better); refuted if the vector index wins by >25% relative

## Outputs
- `reports/probe-eval-r08h53-<stamp>.json` - per-probe recalls (bm25/vector/union), seed overlap, verdict
- In-notebook: channel comparison table, verdict

In [1]:
# Imports
# stdlib
import datetime  # report stamps
import json  # report persistence
import math  # BM25 idf
import os  # graph selection env
import re  # tokenization + gold matching
from collections import Counter  # term frequencies

# third party
import yaml  # probe set
from pathlib import Path
from rich import print as rprint  # semantic output
from rich.progress import Progress  # probe loop

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j2:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

# project
from knowledge_graph_foundry import Foundry, load_settings  # pipeline
from knowledge_graph_foundry.extraction import generate_embeddings  # Titan probe embedding
from knowledge_graph_foundry.graph.graphrag import vector_query  # dense channel
from knowledge_graph_foundry.models import Entity  # embedding probe wrapper

2026-07-06 20:48:45.071 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# Configuration
PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")  # 28 probes with gold evidence

settings = load_settings(Path("../config.yml"))
TOP_K = settings.graphrag.top_k                  # production seed budget, both channels
VEC_INDEX = settings.graphrag.vector_index_name  # entity vector index
BM25_K1 = 1.5                                    # Okapi term-frequency saturation
BM25_B = 0.75                                    # Okapi length normalization

# pre-registered acceptance bar (experiments log, R08-H53)
BAR_WITHIN = 0.10   # confirmed if BM25 within 10% relative (or better)
BAR_REFUTE = 0.25   # refuted if vector beats BM25 by >25% relative

probes = yaml.safe_load(PROBES_PATH.read_text())
gold_probes = [p for p in probes if p.get("gold_evidence")]

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
[bold]Graph[/bold]
  Neo4j: [cyan]{os.environ['NEO4J_URI']}[/cyan]  Vector index: [cyan]{VEC_INDEX}[/cyan]

[bold]Channels[/bold]
  Seed budget k: [yellow]{TOP_K}[/yellow] [dim](both channels, production value)[/dim]
  BM25: k1 [yellow]{BM25_K1}[/yellow], b [yellow]{BM25_B}[/yellow], corpus = embedding text verbatim

[bold]Probes[/bold]
  Set: [cyan]{PROBES_PATH}[/cyan] ([yellow]{len(gold_probes)}[/yellow] with gold evidence)

[bold]Acceptance bar[/bold]
  Confirmed: BM25 within [yellow]{BAR_WITHIN:.0%}[/yellow] relative of vector (or better)
  Refuted: vector wins by >[yellow]{BAR_REFUTE:.0%}[/yellow] relative
""")


def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())


# gold matcher - the H34 iteration-3 conventions
_UNIT = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"


def present(gold, ctx_norm):
    ng = _norm(gold)
    if ng in ctx_norm:
        return True
    squashed = re.sub(r"[\s,()]", "", ctx_norm)
    skeleton = re.sub(r"[\s,()]", "", re.sub(_UNIT, "", ng))
    if any(ch.isdigit() for ch in skeleton) and len(skeleton) >= 5 and skeleton in squashed:
        return True
    tokens = re.findall(r"[\w.\-/]*\d[\w.\-/]*", gold)
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ctx_norm or re.sub(r"[\s,()]", "", _norm(t)) in squashed)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng))
    ctx_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx_norm))
    return bool(words) and len(words & ctx_words) / len(words) >= 0.6

Configuration
────────────────────────────────────────
Graph
  Neo4j: bolt://user-konrad.jelen-kgf-neo4j2:7687  Vector index: kgf_entity_embeddings

Channels
  Seed budget k: 8 (both channels, production value)
  BM25: k1 1.5, b 0.75, corpus = embedding text verbatim

Probes
  Set: ../tests/probes/cpap-probe-set.yml (24 with gold evidence)

Acceptance bar
  Confirmed: BM25 within 10% relative of vector (or better)
  Refuted: vector wins by >25% relative

## BM25 index over the embedding text

One BM25 document per entity, built from the exact string the Titan embedding encoded (`{type}: {name} - {description[:200]}`) - the comparison is between retrieval FUNCTIONS over identical information, not between corpora. Plain Okapi, alphanumeric casefold tokens.

In [3]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.casefold())


class BM25:
    def __init__(self, docs, k1=BM25_K1, b=BM25_B):
        self.k1, self.b = k1, b
        self.tf = [Counter(tokenize(d)) for d in docs]
        self.dl = [sum(c.values()) for c in self.tf]
        self.avgdl = sum(self.dl) / len(self.dl)
        df = Counter()
        for c in self.tf:
            df.update(c.keys())
        n = len(docs)
        self.idf = {t: math.log(1 + (n - d + 0.5) / (d + 0.5)) for t, d in df.items()}

    def top_k(self, query, k):
        q = tokenize(query)
        scores = []
        for i, c in enumerate(self.tf):
            s = 0.0
            for t in q:
                if t in c:
                    f = c[t]
                    s += self.idf[t] * f * (self.k1 + 1) / (
                        f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                    )
            scores.append(s)
        order = sorted(range(len(scores)), key=lambda i: -scores[i])
        return [i for i in order[:k] if scores[i] > 0]


with Foundry(settings) as f:
    with f.driver.session() as s:
        rows = s.run(
            "MATCH (e:Entity) WHERE e.name IS NOT NULL "
            "RETURN e.id AS id, e.name AS name, labels(e) AS types, "
            "coalesce(e.description, '') AS description"
        ).data()

ids = [r["id"] for r in rows]
types_first = [
    next((t for t in r["types"] if t != "Entity"), "Entity") for r in rows
]
docs = [f"{t}: {r['name']} - {r['description'][:200]}" for t, r in zip(types_first, rows)]
bm25 = BM25(docs)
rprint(f"BM25 index: [yellow]{len(docs)}[/yellow] entities, avg doc length [yellow]{bm25.avgdl:.1f}[/yellow] tokens")

BM25 index: 2798 entities, avg doc length 11.7 tokens

## Head-to-head seeding, union channel, and verdict

Per probe: BM25 top-8 vs vector top-8 (one Titan call per probe for the dense side), both rendered with the H34 pure-seed convention (node's own description + properties + relations, alias merge off). Union = the two seed sets concatenated (2k budget - complementarity check, not a matched-budget claim). Recall per probe, means compared against the pre-registered bar.

In [4]:
def render_nodes(session, node_ids):
    """H34 pure-seed render: description + prop_* spec + currently-valid
    relations, NO alias merge (that is a separate channel, not the seed)."""
    blocks = []
    for nid in node_ids:
        row = session.run(
            "MATCH (e:Entity {id: $id}) RETURN e.name AS name, labels(e) AS types, "
            "e.description AS description, properties(e) AS props",
            id=nid,
        ).single()
        if row is None:
            continue
        spec = {k.removeprefix("prop_"): v for k, v in row["props"].items() if k.startswith("prop_")}
        rels = session.run(
            "MATCH (e:Entity {id: $id})-[r]-(n:Entity) "
            "WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' "
            "RETURN type(r) AS rel, n.name AS name LIMIT 15",
            id=nid,
        ).data()
        blocks.append(
            f"## {row['name']} ({', '.join(row['types'])})\n{row['description'] or ''}\n"
            f"Properties: {json.dumps(spec, default=str)}\n"
            "Relations: " + "; ".join(f"{r['rel']} -> {r['name']}" for r in rels)
        )
    return _norm("\n".join(blocks))


def rrf_fuse(a, b, kc=60, topk=TOP_K):
    """Reciprocal-rank fusion of the two channel rankings at MATCHED budget -
    the union channel above spends 2k seeds; this spends the production k."""
    scores = {}
    for lst in (a, b):
        for rank, i in enumerate(lst):
            scores[i] = scores.get(i, 0.0) + 1.0 / (kc + rank + 1)
    return sorted(scores, key=lambda i: -scores[i])[:topk]


per_probe = {}
with Foundry(settings) as f, Progress() as progress:
    task = progress.add_task("BM25 vs Titan", total=len(gold_probes))
    for p in gold_probes:
        q = p["question"]
        bm_ids = [ids[i] for i in bm25.top_k(q, TOP_K)]
        probe_e = Entity.create(q[:80], types=["Query"], description=q)
        emb = generate_embeddings([probe_e], settings.embeddings)[0].embedding
        vec16 = [s["id"] for s in vector_query(f.driver, emb, VEC_INDEX, top_k=2 * TOP_K)]
        vec_ids = vec16[:TOP_K]
        union_ids = list(dict.fromkeys(vec_ids + bm_ids))
        rrf_ids = rrf_fuse(vec_ids, bm_ids)
        with f.driver.session() as session:
            ctx_bm = render_nodes(session, bm_ids)
            ctx_vec = render_nodes(session, vec_ids)
            ctx_union = render_nodes(session, union_ids)
            ctx_rrf = render_nodes(session, rrf_ids)
            ctx_vec16 = render_nodes(session, vec16)  # budget control for the union claim
        golds = p["gold_evidence"]
        per_probe[p["id"]] = {
            "recall_bm25": sum(present(g, ctx_bm) for g in golds) / len(golds),
            "recall_vector": sum(present(g, ctx_vec) for g in golds) / len(golds),
            "recall_union": sum(present(g, ctx_union) for g in golds) / len(golds),
            "recall_rrf": sum(present(g, ctx_rrf) for g in golds) / len(golds),
            "recall_vec16": sum(present(g, ctx_vec16) for g in golds) / len(golds),
            "seed_overlap": len(set(bm_ids) & set(vec_ids)),
            "n_bm25_seeds": len(bm_ids),
        }
        progress.advance(task)

n = len(per_probe)
mean_bm = sum(v["recall_bm25"] for v in per_probe.values()) / n
mean_vec = sum(v["recall_vector"] for v in per_probe.values()) / n
mean_union = sum(v["recall_union"] for v in per_probe.values()) / n
mean_rrf = sum(v["recall_rrf"] for v in per_probe.values()) / n
mean_vec16 = sum(v["recall_vec16"] for v in per_probe.values()) / n
mean_overlap = sum(v["seed_overlap"] for v in per_probe.values()) / n
rel_gap = (mean_vec - mean_bm) / mean_vec if mean_vec else 0.0

bm_wins = [pid for pid, v in per_probe.items() if v["recall_bm25"] > v["recall_vector"]]
vec_wins = [pid for pid, v in per_probe.items() if v["recall_vector"] > v["recall_bm25"]]

verdict = (
    "REFUTED" if rel_gap > BAR_REFUTE
    else "CONFIRMED" if rel_gap <= BAR_WITHIN
    else "INCONCLUSIVE"
)

rprint(f"""[bold cyan]BM25 vs Titan - pure-seed evidence recall[/bold cyan]
[dim]{"\u2500" * 40}[/dim]
  BM25:   [yellow]{mean_bm:.3f}[/yellow]
  Vector: [yellow]{mean_vec:.3f}[/yellow]  [dim](relative gap {rel_gap:+.1%})[/dim]
  Union:  [yellow]{mean_union:.3f}[/yellow]  [dim](2k budget - complementarity check)[/dim]
  RRF@k:  [yellow]{mean_rrf:.3f}[/yellow]  [dim](matched k={TOP_K} budget - the shippable hybrid)[/dim]
  Vec@2k: [yellow]{mean_vec16:.3f}[/yellow]  [dim](budget control: is the union gain fusion or just budget?)[/dim]
  Mean seed overlap: [yellow]{mean_overlap:.1f}[/yellow]/{TOP_K}
  Probe wins - BM25: [yellow]{bm_wins}[/yellow]
  Probe wins - vector: [yellow]{vec_wins}[/yellow]
  Verdict: [{'green' if verdict == 'CONFIRMED' else 'red' if verdict == 'REFUTED' else 'yellow'}]{verdict}[/]
""")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"probe-eval-r08h53-{stamp}.json"
out.write_text(json.dumps({
    "mean_bm25": mean_bm, "mean_vector": mean_vec, "mean_union": mean_union,
    "mean_rrf": mean_rrf, "mean_vec16": mean_vec16,
    "relative_gap": rel_gap, "mean_seed_overlap": mean_overlap,
    "bm25_wins": bm_wins, "vector_wins": vec_wins,
    "per_probe": per_probe, "verdict": verdict,
}, indent=2))
rprint("saved", str(out))

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-06 20:48:45.969 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-06 20:48:45.971 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

BM25 vs Titan - pure-seed evidence recall
────────────────────────────────────────
  BM25:   0.583
  Vector: 0.667  (relative gap +12.5%)
  Union:  0.750  (2k budget - complementarity check)
  RRF@k:  0.646  (matched k=8 budget - the shippable hybrid)
  Vec@2k: 0.854  (budget control: is the union gain fusion or just budget?)
  Mean seed overlap: 2.5/8
  Probe wins - BM25: ['P01', 'P19']
  Probe wins - vector: ['P04', 'P12', 'P21', 'P24']
  Verdict: INCONCLUSIVE

saved ../reports/probe-eval-r08h53-20260706-184855.json